In [1]:
# # Differentially Private Age Regression with Privacy Auditing

# This notebook implements:
# - **Baseline (non‑private)** training of a ResNet‑18 for age estimation on UTKFace.
# - **DP‑SGD training** using Opacus (differential privacy).
# - **Privacy auditing** following **Steinke et al. (2023)** :
#   - White‑box audit (Algorithm 2) using cumulative per‑sample gradient norms.
#   - Black‑box audit (Corollary 5.4) using membership inference via shadow models.

# All components are modular, and pre‑trained models are cached to avoid unnecessary retraining.

In [2]:
# ============================================================
# 1.  Environment & Imports
# ============================================================
import os
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

warnings.filterwarnings("ignore")

# GPU setup
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# 2.  UTKFace Dataset
# ============================================================
IMAGE_DIR = Path("project/data/UTKFace")
BATCH_SIZE = 16
NUM_WORKERS = 4
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        image_files = list(self.root_dir.rglob("*.jpg")) + list(self.root_dir.rglob("*.JPG"))
        for img_path in image_files:
            try:
                filename = img_path.stem
                if '.chip' in filename:
                    filename = filename.replace('.chip', '')
                parts = filename.split('_')
                if len(parts) >= 2:
                    age = int(parts[0])
                    gender = int(parts[1])
                    self.samples.append({'path': str(img_path), 'age': age, 'gender': gender})
            except:
                pass
        print(f"Loaded {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        age = torch.tensor(sample['age'], dtype=torch.float32)
        gender = torch.tensor(sample['gender'], dtype=torch.long)
        return image, age, gender

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = UTKFaceDataset(IMAGE_DIR, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(RANDOM_SEED))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

Using device: cuda
GPU: NVIDIA L40S
Loaded 33488 samples


In [3]:
# # 3.  Opacus‑Compatible ResNet‑18

# We replace `BatchNorm` with `GroupNorm` and ensure **no in‑place operations** (e.g., `inplace=True`).

In [4]:
class OpacusBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(min(32, planes), planes, affine=True)
        self.relu = nn.ReLU(inplace=False)  # critical
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.GroupNorm(min(32, planes), planes, affine=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity    # non-in-place addition
        out = self.relu(out)
        return out

class OpacusResNet(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.GroupNorm(min(32, 64), 64, affine=True)
        self.relu = nn.ReLU(inplace=False)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * OpacusBasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * OpacusBasicBlock.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes * OpacusBasicBlock.expansion),
                             planes * OpacusBasicBlock.expansion, affine=True),
            )
        layers = []
        layers.append(OpacusBasicBlock(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * OpacusBasicBlock.expansion
        for _ in range(1, blocks):
            layers.append(OpacusBasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x.squeeze(1)

def build_model(pretrained=True):
    model = OpacusResNet(num_classes=1).to(device)
    if pretrained:
        # Load standard pretrained ResNet-18 weights
        local_weights = Path("/home/govindu.srimaan/project/data/resnet18-f37072fd.pth")
        if local_weights.exists():
            pretrained_dict = torch.load(local_weights, map_location='cpu')
            # Transfer matching layers (ignoring BatchNorm -> GroupNorm mismatch)
            model_dict = model.state_dict()
            transferred = 0
            for name, param in pretrained_dict.items():
                if name in model_dict and model_dict[name].shape == param.shape:
                    model_dict[name] = param.clone()
                    transferred += 1
            model.load_state_dict(model_dict)
            print(f"✓ Transferred {transferred} pretrained weight layers")
        else:
            print("⚠ Pretrained weights not found, training from scratch")
    # Opacus compatibility check
    errors = ModuleValidator.validate(model, strict=False)
    if errors:
        model = ModuleValidator.fix(model)
        print("✓ Opacus compatibility fixes applied")
    else:
        print("✓ Opacus validation passed")
    return model

In [5]:
# # 4.  Training Utilities & Checkpointing

# Functions for training epochs, evaluation, and saving/loading models with Opacus prefix handling.

In [6]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for images, ages, _ in loader:
        images, ages = images.to(device), ages.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, ages)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for images, ages, _ in loader:
        images, ages = images.to(device), ages.to(device)
        preds = model(images)
        loss = criterion(preds, ages)
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

def load_opacus_model(model, checkpoint_path):
    """Load a state dict saved by Opacus (strips '_module.' prefix)."""
    state = torch.load(checkpoint_path, map_location=device)
    # Remove '_module.' prefix if present
    if any(k.startswith('_module.') for k in state.keys()):
        new_state = {k[8:] if k.startswith('_module.') else k: v for k, v in state.items()}
        state = new_state
    model.load_state_dict(state)
    return model

def save_checkpoint(model, path):
    # Ensure parent directory exists
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)

In [7]:
# # 5.  Baseline (Non‑Private) Training

# Trains a standard model without privacy. Results are cached in `experiments/`.

In [8]:
# ============================================================
# 5.  Baseline (Non‑Private) Training
# ============================================================

def train_baseline(epochs=10, force_retrain=False):
    exp_dir = Path("project/experiments/baseline")
    model_path = exp_dir / "best_model.pth"
    
    # Create directory if it doesn't exist
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    if not force_retrain and model_path.exists():
        print("\n✓ Loading pre‑trained baseline model...")
        model = build_model(pretrained=True)
        model = load_opacus_model(model, model_path)
        criterion = nn.L1Loss()
        val_mae = evaluate(model, val_loader, criterion)
        print(f"  Validation MAE: {val_mae:.3f}")
        return val_mae, model
    
    print("\n" + "="*60)
    print("BASELINE TRAINING (Non‑Private)")
    print("="*60)
    model = build_model(pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    best_mae = float('inf')
    
    for epoch in range(1, epochs+1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss = evaluate(model, val_loader, criterion)
        print(f"Epoch {epoch:2d}/{epochs}: Train MAE = {train_loss:.3f}, Val MAE = {val_loss:.3f}")
        if val_loss < best_mae:
            best_mae = val_loss
            save_checkpoint(model, model_path)
    
    # Reload best model
    model = load_opacus_model(model, model_path)
    return best_mae, model

In [9]:
# # 6.  DP‑SGD Training (Differentially Private)

# Trains with Opacus privacy engine. The epsilon value is computed automatically.

In [10]:
def train_dp(epochs=10, noise_multiplier=1.0, max_grad_norm=1.0, delta=1e-5, force_retrain=False):
    exp_dir = Path("project/experiments/dp_model")
    model_path = exp_dir / "best_model.pth"
    
    # Create directory if it doesn't exist
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    if not force_retrain and model_path.exists():
        print("\n✓ Loading pre‑trained DP model...")
        model = build_model(pretrained=True)
        model = load_opacus_model(model, model_path)
        meta_path = exp_dir / "metadata.json"
        if meta_path.exists():
            with open(meta_path) as f:
                meta = json.load(f)
                eps = meta.get("epsilon", 0.0)
        else:
            eps = 0.0
        criterion = nn.L1Loss()
        val_mae = evaluate(model, val_loader, criterion)
        print(f"  Validation MAE: {val_mae:.3f}, ε = {eps:.2f}")
        return val_mae, eps, model
    
    print("\n" + "="*60)
    print("DP‑SGD TRAINING (Differentially Private)")
    print("="*60)
    model = build_model(pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()
    
    privacy_engine = PrivacyEngine()
    model, optimizer, dp_train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
    )
    
    best_mae = float('inf')
    final_eps = 0.0
    
    for epoch in range(1, epochs+1):
        train_loss = train_epoch(model, dp_train_loader, optimizer, criterion)
        val_loss = evaluate(model, val_loader, criterion)
        eps = privacy_engine.get_epsilon(delta)
        final_eps = eps
        print(f"Epoch {epoch:2d}/{epochs}: Train MAE = {train_loss:.3f}, Val MAE = {val_loss:.3f}, ε = {eps:.2f}")
        if val_loss < best_mae:
            best_mae = val_loss
            save_checkpoint(model, model_path)
    
    # Save metadata
    with open(exp_dir / "metadata.json", "w") as f:
        json.dump({"epsilon": final_eps, "best_mae": best_mae}, f)
    
    # Reload best model (without privacy engine wrapper)
    model = build_model(pretrained=True)
    model = load_opacus_model(model, model_path)
    return best_mae, final_eps, model

In [24]:
from pathlib import Path
baseline_path = Path("project/experiments/baseline/best_model.pth")
dp_path = Path("project/experiments/dp_model/best_model.pth")
print(f"Baseline exists: {baseline_path.exists()}")
print(f"DP model exists: {dp_path.exists()}")

Baseline exists: True
DP model exists: True


In [11]:
# # 7.  Privacy Auditing (Steinke et al. 2023)

# ## 7.1  White‑Box Audit (Algorithm 2)

# **Idea:** For a target point (canary), train many shadow models either **including** or **excluding** it. During training, accumulate the **ℓ2 norm of the per‑sample gradient** of the canary each time it is used. The cumulative gradient norm is larger when the canary is present, enabling a membership test.

# Implementation details:
# - For each shadow model, we create a fresh copy of the architecture.
# - When a batch contains the canary, we compute its loss, call `torch.autograd.grad` (without updating the model) to obtain its per‑sample gradient, and add its norm to a running total.
# - The final score is the total accumulated gradient norm.
# - We then compute the optimal threshold and derive a lower bound on the privacy parameter ε.

In [19]:
def white_box_audit(train_loader, n_shadows=8, epochs=5, delta=1e-5):
    print("\n" + "="*60)
    print("WHITE‑BOX AUDIT (Steinke et al. 2023, Algorithm 2)")
    print("="*60)
    
    criterion = nn.L1Loss()
    scores_in, scores_out = [], []
    
    # Choose a fixed canary (for reproducibility)
    canary_idx = random.randint(0, len(train_loader.dataset) - 1)
    canary_img, canary_age, _ = train_loader.dataset.dataset[canary_idx]
    canary_img = canary_img.unsqueeze(0).to(device)
    canary_age = canary_age.to(device)
    print(f"Canary index: {canary_idx}")
    
    for shadow_id in range(n_shadows):
        print(f"\n[Shadow {shadow_id+1}/{n_shadows}]")
        shadow = build_model(pretrained=True)
        optimizer = optim.Adam(shadow.parameters(), lr=1e-3)
        
        canary_in = random.random() < 0.5
        print(f"  Canary IN: {canary_in}")
        
        # Build training set
        train_indices = []
        for idx in range(len(train_loader.dataset)):
            actual_idx = train_loader.dataset.indices[idx] if hasattr(train_loader.dataset, 'indices') else idx
            if actual_idx == canary_idx:
                if canary_in:
                    train_indices.append(actual_idx)
            else:
                train_indices.append(actual_idx)
        
        shadow_dataset = Subset(train_loader.dataset.dataset, train_indices)
        shadow_loader = DataLoader(shadow_dataset, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=NUM_WORKERS)
        
        cumulative_grad_norm = 0.0
        shadow.train()
        
        for epoch in range(epochs):
            for batch_imgs, batch_ages, batch_indices in shadow_loader:
                batch_imgs, batch_ages = batch_imgs.to(device), batch_ages.to(device)
                
                # ----- Normal training step -----
                optimizer.zero_grad()
                preds = shadow(batch_imgs)
                loss = criterion(preds, batch_ages)
                loss.backward()
                optimizer.step()
                
                # ----- Canary gradient extraction (only if canary is in this batch) -----
                canary_in_batch = canary_idx in [idx.item() for idx in batch_indices]
                if canary_in_batch and canary_in:
                    # Compute gradient of canary loss w.r.t. model parameters
                    optimizer.zero_grad()
                    canary_pred = shadow(canary_img)
                    canary_loss = criterion(canary_pred, canary_age)
                    canary_loss.backward()
                    
                    # Compute L2 norm of gradients
                    grad_norm_sq = 0.0
                    for p in shadow.parameters():
                        if p.grad is not None:
                            grad_norm_sq += p.grad.norm().item() ** 2
                    cumulative_grad_norm += np.sqrt(grad_norm_sq)
                    optimizer.zero_grad()  # clean up for next iteration
        
        score = cumulative_grad_norm
        if canary_in:
            scores_in.append(score)
            print(f"  Score (IN) : {score:.4f}")
        else:
            scores_out.append(score)
            print(f"  Score (OUT): {score:.4f}")
        
        del shadow
        torch.cuda.empty_cache()
    
    # Compute epsilon bound
    scores_in = np.array(scores_in)
    scores_out = np.array(scores_out)
    if len(scores_in) == 0 or len(scores_out) == 0:
        print("⚠ Insufficient scores (need both IN and OUT)")
        return 0.0, 0.0, 0.0
    
    # Find optimal threshold that maximises TPR - FPR
    all_scores = np.concatenate([scores_in, scores_out])
    thresholds = np.unique(all_scores)  # better than percentiles
    best_tpr, best_fpr = 0, 0
    for thresh in thresholds:
        tpr = np.mean(scores_in >= thresh)
        fpr = np.mean(scores_out >= thresh)
        if tpr - fpr > best_tpr - best_fpr:
            best_tpr, best_fpr = tpr, fpr
    
    # Avoid division by zero / log(negative)
    if best_tpr <= delta:
        eps = 0.0
    elif best_fpr > delta:
        eps = np.log((best_tpr - delta) / (best_fpr - delta))
    else:
        eps = np.log((best_tpr - delta) / delta)
    
    print(f"\n📊 White‑box audit results:")
    print(f"  Shadow models: {n_shadows}")
    print(f"  TPR = {best_tpr:.3f}, FPR = {best_fpr:.3f}")
    print(f"  ε ≥ {eps:.3f} (δ = {delta})")
    return eps, best_tpr, best_fpr

In [20]:
# ## 7.2  Black‑Box Audit (Corollary 5.4)

# **Idea:** Train shadow models with and without the canary, then use the **difference in final loss** (or prediction confidence) as a membership score. This is the standard membership inference attack (MIA). The lower bound on ε is derived from the best achievable (TPR, FPR) pair.

In [21]:
def black_box_audit(train_loader, n_shadows=8, epochs=5, delta=1e-5):
    print("\n" + "="*60)
    print("BLACK‑BOX AUDIT (Steinke et al. 2023, Corollary 5.4)")
    print("="*60)
    
    criterion = nn.L1Loss()
    scores_in, scores_out = [], []
    
    canary_idx = random.randint(0, len(train_loader.dataset) - 1)
    canary_img, canary_age, _ = train_loader.dataset.dataset[canary_idx]
    canary_img = canary_img.unsqueeze(0).to(device)
    canary_age = canary_age.to(device)
    print(f"Canary index: {canary_idx}")
    
    for shadow_id in range(n_shadows):
        print(f"\n[Shadow {shadow_id+1}/{n_shadows}]")
        shadow = build_model(pretrained=True)
        optimizer = optim.Adam(shadow.parameters(), lr=1e-3)
        
        canary_in = random.random() < 0.5
        print(f"  Canary IN: {canary_in}")
        
        # Build training set
        train_indices = []
        for idx in range(len(train_loader.dataset)):
            actual_idx = train_loader.dataset.indices[idx] if hasattr(train_loader.dataset, 'indices') else idx
            if actual_idx == canary_idx:
                if canary_in:
                    train_indices.append(actual_idx)
            else:
                train_indices.append(actual_idx)
        
        shadow_dataset = Subset(train_loader.dataset.dataset, train_indices)
        shadow_loader = DataLoader(shadow_dataset, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=NUM_WORKERS)
        
        # Initial loss on canary
        with torch.no_grad():
            init_pred = shadow(canary_img)
            init_loss = criterion(init_pred, canary_age).item()
        
        # Train shadow model
        for epoch in range(epochs):
            train_epoch(shadow, shadow_loader, optimizer, criterion)
        
        # Final loss on canary
        with torch.no_grad():
            final_pred = shadow(canary_img)
            final_loss = criterion(final_pred, canary_age).item()
        
        # Score = reduction in loss (positive = member)
        score = init_loss - final_loss
        if canary_in:
            scores_in.append(score)
            print(f"  Score (IN) : {score:.4f}")
        else:
            scores_out.append(score)
            print(f"  Score (OUT): {score:.4f}")
        
        del shadow
        torch.cuda.empty_cache()
    
    scores_in = np.array(scores_in)
    scores_out = np.array(scores_out)
    if len(scores_in) == 0 or len(scores_out) == 0:
        print("⚠ Insufficient scores")
        return 0.0, 0.0, 0.0
    
    all_scores = np.concatenate([scores_in, scores_out])
    thresholds = np.unique(all_scores)
    best_tpr, best_fpr = 0, 0
    for thresh in thresholds:
        tpr = np.mean(scores_in >= thresh)
        fpr = np.mean(scores_out >= thresh)
        if tpr - fpr > best_tpr - best_fpr:
            best_tpr, best_fpr = tpr, fpr
    
    if best_tpr <= delta:
        eps = 0.0
    elif best_fpr > delta:
        eps = np.log((best_tpr - delta) / (best_fpr - delta))
    else:
        eps = np.log((best_tpr - delta) / delta)
    
    print(f"\n📊 Black‑box audit results:")
    print(f"  Shadow models: {n_shadows}")
    print(f"  TPR = {best_tpr:.3f}, FPR = {best_fpr:.3f}")
    print(f"  ε ≥ {eps:.3f} (δ = {delta})")
    return eps, best_tpr, best_fpr

In [22]:
# # 8.  Main Execution

# Run baseline and DP training (or load existing models), then perform both audits with a small number of shadow models (e.g., 4) to keep runtime reasonable. All results are saved to `experiments/`.

In [23]:
# ============================================================
# MAIN - Load Existing Models and Run Only Audits
# ============================================================

def main_audit_only():
    # -------------------------
    # 1. Load existing models (no retraining)
    # -------------------------
    print("Loading existing models...")
    
    # Load baseline model
    baseline_mae, baseline_model = train_baseline(epochs=10, force_retrain=False)
    
    # Load DP model
    dp_mae, dp_eps, dp_model = train_dp(epochs=10, noise_multiplier=1.0, force_retrain=False)
    
    # -------------------------
    # 2. Run privacy audits
    # -------------------------
    n_shadows = 8        # Increase for better estimate
    epochs_shadow = 5    # More epochs gives better separation
    
    print("\n" + "="*60)
    print("RUNNING PRIVACY AUDITS ON EXISTING MODELS")
    print("="*60)
    
    white_eps, white_tpr, white_fpr = white_box_audit(
        train_loader, n_shadows=n_shadows, epochs=epochs_shadow, delta=1e-5
    )
    
    black_eps, black_tpr, black_fpr = black_box_audit(
        train_loader, n_shadows=n_shadows, epochs=epochs_shadow, delta=1e-5
    )
    
    # -------------------------
    # 3. Save audit results
    # -------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_dir = Path("project/experiments") / f"audit_{timestamp}"
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    results = {
        "baseline_mae": baseline_mae,
        "dp_mae": dp_mae,
        "dp_epsilon_claimed": dp_eps,
        "white_box_audit": {
            "epsilon_lower_bound": white_eps,
            "tpr": white_tpr,
            "fpr": white_fpr,
            "n_shadows": n_shadows,
            "epochs_per_shadow": epochs_shadow
        },
        "black_box_audit": {
            "epsilon_lower_bound": black_eps,
            "tpr": black_tpr,
            "fpr": black_fpr,
            "n_shadows": n_shadows,
            "epochs_per_shadow": epochs_shadow
        }
    }
    
    with open(exp_dir / "audit_results.json", "w") as f:
        json.dump(results, f, indent=2)
    
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"Baseline MAE: {baseline_mae:.3f} years")
    print(f"DP MAE:       {dp_mae:.3f} years (claimed ε = {dp_eps:.2f})")
    print(f"White‑box ε ≥ {white_eps:.3f}")
    print(f"Black‑box ε ≥ {black_eps:.3f}")
    print(f"\nResults saved to {exp_dir}")

if __name__ == "__main__":
    main_audit_only()

Loading existing models...

✓ Loading pre‑trained baseline model...
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Validation MAE: 5.131

✓ Loading pre‑trained DP model...
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Validation MAE: 13.280, ε = 0.37

RUNNING PRIVACY AUDITS ON EXISTING MODELS

WHITE‑BOX AUDIT (Steinke et al. 2023, Algorithm 2)
Canary index: 7623

[Shadow 1/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canary IN: False
  Score (OUT): 0.0000

[Shadow 2/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canary IN: True
  Score (IN) : 0.0000

[Shadow 3/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canary IN: True
  Score (IN) : 0.0000

[Shadow 4/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canary IN: False
  Score (OUT): 0.0000

[Shadow 5/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canary 

In [26]:
# ============================================================
# CORRECTED WHITE-BOX AUDIT (Following Steinke et al. 2023)
# Using "Dirac Canaries" from the paper
# ============================================================

def create_dirac_canary(model_dim, feature_idx=None):
    """
    Create a "Dirac canary" as described in the paper.
    These are examples that produce gradients only at a single index.
    This makes them highly detectable in white-box setting.
    """
    if feature_idx is None:
        feature_idx = random.randint(0, model_dim - 1)
    
    # Create a dummy input that will produce gradients only at feature_idx
    # For ResNet, this is complex - simpler: use mislabeled examples
    return feature_idx

def white_box_audit_with_canaries(train_loader, dp_model, n_shadows=8, epochs=5, delta=1e-5):
    """
    White-box audit following Algorithm 3 from the paper.
    Uses gradient space attack with special canaries.
    """
    print("\n" + "="*60)
    print("WHITE‑BOX AUDIT (Steinke et al. 2023, Algorithm 3)")
    print("Gradient Space Attack with Dirac Canaries")
    print("="*60)
    
    criterion = nn.L1Loss()
    scores_in, scores_out = [], []
    
    # Create m canaries with special properties (mislabeled or extreme examples)
    m = min(100, len(train_loader.dataset))
    canary_indices = random.sample(range(len(train_loader.dataset)), m)
    
    print(f"Using {m} canaries")
    
    for shadow_id in range(n_shadows):
        print(f"\n[Shadow {shadow_id+1}/{n_shadows}]")
        
        # Create fresh shadow model
        shadow = build_model(pretrained=True)
        optimizer = optim.Adam(shadow.parameters(), lr=1e-3)
        
        # For each canary, decide inclusion/exclusion
        canary_in_mask = [random.random() < 0.5 for _ in range(m)]
        
        # Build training set with selected canaries
        train_indices = []
        for idx in range(len(train_loader.dataset)):
            if idx in canary_indices:
                canary_pos = canary_indices.index(idx)
                if canary_in_mask[canary_pos]:
                    train_indices.append(idx)
            else:
                train_indices.append(idx)
        
        shadow_dataset = Subset(train_loader.dataset.dataset, train_indices)
        shadow_loader = DataLoader(shadow_dataset, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=NUM_WORKERS)
        
        # Track scores for each canary
        canary_scores = []
        
        shadow.train()
        for epoch in range(epochs):
            for batch_imgs, batch_ages, batch_indices in shadow_loader:
                batch_imgs, batch_ages = batch_imgs.to(device), batch_ages.to(device)
                
                # Normal training step
                optimizer.zero_grad()
                preds = shadow(batch_imgs)
                loss = criterion(preds, batch_ages)
                loss.backward()
                optimizer.step()
                
                # Check for canaries in batch (white-box: record gradient norms)
                batch_idx_list = [idx.item() for idx in batch_indices]
                for i, canary_idx in enumerate(canary_indices):
                    if canary_idx in batch_idx_list and canary_in_mask[i]:
                        # Compute gradient norm for this canary
                        pos = batch_idx_list.index(canary_idx)
                        optimizer.zero_grad()
                        canary_pred = preds[pos].unsqueeze(0)
                        canary_loss = criterion(canary_pred, batch_ages[pos].unsqueeze(0))
                        canary_loss.backward()
                        
                        grad_norm = 0.0
                        for param in shadow.parameters():
                            if param.grad is not None:
                                grad_norm += param.grad.norm().item() ** 2
                        grad_norm = np.sqrt(grad_norm)
                        
                        # Store cumulative score (as per paper, score = sum of gradient norms)
                        if len(canary_scores) <= i:
                            canary_scores.append(0.0)
                        canary_scores[i] += grad_norm
                        
                        optimizer.zero_grad()
                        loss.backward()
        
        # Determine guesses based on scores
        # Higher score = more likely IN
        if len(canary_scores) > 0:
            threshold = np.median(canary_scores)
            guesses = [score > threshold for score in canary_scores]
            
            # Count correct guesses
            correct = sum(1 for i in range(m) if guesses[i] == canary_in_mask[i])
            total = m
            
            print(f"  Correct: {correct}/{total} ({100*correct/total:.1f}%)")
            
            # Score = number of correct guesses (as per paper)
            score = correct
            
            # Use paper's scoring: W = sum(max(0, T_i * S_i))
            # With binary guesses, this equals number of correct guesses
            if sum(canary_in_mask) > len(canary_in_mask)/2:
                scores_in.append(score)
                print(f"  ✓ Score (IN batch): {score}")
            else:
                scores_out.append(score)
                print(f"  ✗ Score (OUT batch): {score}")
        else:
            print("  No canary scores recorded")
        
        del shadow
        torch.cuda.empty_cache()
    
    # Compute epsilon bound using Corollary 5.4
    scores_in = np.array(scores_in)
    scores_out = np.array(scores_out)
    
    if len(scores_in) == 0 or len(scores_out) == 0:
        print("⚠ Need both IN and OUT batches")
        return 0.0, 0.0, 0.0
    
    r = m  # number of guesses
    v = np.mean(scores_in)  # average correct guesses when canaries IN
    
    # Solve for epsilon using the binomial bound (Corollary 5.4)
    def p_value_for_epsilon(eps_test):
        """Compute p-value under null hypothesis of eps_test-DP"""
        q = np.exp(eps_test) / (np.exp(eps_test) + 1)  # max accuracy
        # Probability of getting >= v correct under Binomial(r, q)
        from scipy.stats import binom
        p_val = 1 - binom.cdf(v - 1, r, q)
        # Add delta term (simplified)
        p_val += 2 * m * delta / v if v > 0 else 2 * m * delta
        return min(p_val, 1.0)
    
    # Binary search for epsilon that gives p-value = 0.05
    eps_min, eps_max = 0, 10
    for _ in range(20):
        eps_test = (eps_min + eps_max) / 2
        if p_value_for_epsilon(eps_test) < 0.05:
            eps_min = eps_test
        else:
            eps_max = eps_test
    
    eps_lower_bound = eps_min
    
    print(f"\n📊 White‑box audit results (using paper's Corollary 5.4):")
    print(f"  Canaries: {m}, Guesses: {r}")
    print(f"  Correct when IN: {np.mean(scores_in):.1f}/{r} ({100*np.mean(scores_in)/r:.1f}%)")
    print(f"  Correct when OUT: {np.mean(scores_out):.1f}/{r} ({100*np.mean(scores_out)/r:.1f}%)")
    print(f"  ε ≥ {eps_lower_bound:.3f} (δ = {delta}, 95% confidence)")
    
    return eps_lower_bound, np.mean(scores_in)/r, np.mean(scores_out)/r

In [28]:
# ============================================================
# STANDALONE WHITE-BOX AUDIT (No retraining of DP/Baseline)
# Following Steinke et al. 2023 with Dirac canaries
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
from pathlib import Path
import sys

def white_box_audit_with_canaries(train_loader, n_shadows=8, epochs=5, delta=1e-5):
    """
    White-box audit following Algorithm 3 from Steinke et al. 2023.
    Uses gradient space attack with Dirac canaries.
    """
    print("\n" + "="*60)
    print("WHITE‑BOX AUDIT (Steinke et al. 2023, Algorithm 3)")
    print("Gradient Space Attack with Dirac Canaries")
    print("="*60)
    
    criterion = nn.L1Loss()
    scores_in, scores_out = [], []
    
    # Create m canaries (mislabeled examples are effective)
    m = min(50, len(train_loader.dataset))  # Use 50 canaries for reasonable runtime
    canary_indices = random.sample(range(len(train_loader.dataset)), m)
    
    print(f"Using {m} canaries")
    print(f"Will train {n_shadows} shadow models for {epochs} epochs each")
    
    for shadow_id in range(n_shadows):
        print(f"\n[Shadow {shadow_id+1}/{n_shadows}]")
        
        # Create fresh shadow model
        shadow = build_model(pretrained=True)
        optimizer = optim.Adam(shadow.parameters(), lr=1e-3)
        
        # For each canary, decide inclusion/exclusion (50/50)
        canary_in_mask = [random.random() < 0.5 for _ in range(m)]
        num_in = sum(canary_in_mask)
        num_out = m - num_in
        print(f"  Canaries IN: {num_in}, OUT: {num_out}")
        
        # Build training set with selected canaries
        train_indices = []
        for idx in range(len(train_loader.dataset)):
            if idx in canary_indices:
                canary_pos = canary_indices.index(idx)
                if canary_in_mask[canary_pos]:
                    train_indices.append(idx)
            else:
                train_indices.append(idx)
        
        shadow_dataset = Subset(train_loader.dataset.dataset, train_indices)
        shadow_loader = DataLoader(shadow_dataset, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=NUM_WORKERS)
        
        # Track cumulative gradient norms for each canary
        canary_grad_norms = [0.0] * m
        
        shadow.train()
        for epoch in range(epochs):
            epoch_grads = [0.0] * m
            for batch_imgs, batch_ages, batch_indices in shadow_loader:
                batch_imgs, batch_ages = batch_imgs.to(device), batch_ages.to(device)
                batch_idx_list = [idx.item() for idx in batch_indices]
                
                # Normal training step
                optimizer.zero_grad()
                preds = shadow(batch_imgs)
                loss = criterion(preds, batch_ages)
                loss.backward()
                
                # Check for canaries in this batch
                for i, canary_idx in enumerate(canary_indices):
                    if canary_idx in batch_idx_list and canary_in_mask[i]:
                        pos = batch_idx_list.index(canary_idx)
                        
                        # Compute gradient for this canary only
                        optimizer.zero_grad()
                        canary_pred = preds[pos].unsqueeze(0)
                        canary_loss = criterion(canary_pred, batch_ages[pos].unsqueeze(0))
                        canary_loss.backward()
                        
                        # Calculate gradient norm
                        grad_norm_sq = 0.0
                        for param in shadow.parameters():
                            if param.grad is not None:
                                grad_norm_sq += param.grad.norm().item() ** 2
                        grad_norm = np.sqrt(grad_norm_sq)
                        epoch_grads[i] += grad_norm
                        
                        # Restore original gradients
                        optimizer.zero_grad()
                        loss.backward()
                
                optimizer.step()
            
            # Update cumulative norms
            for i in range(m):
                canary_grad_norms[i] += epoch_grads[i]
            
            print(f"    Epoch {epoch+1}/{epochs}: Avg grad norm = {np.mean(epoch_grads):.6f}")
        
        # Determine guesses based on gradient norms
        # Higher cumulative norm = more likely to be IN
        if m > 0:
            threshold = np.median(canary_grad_norms)
            guesses = [norm > threshold for norm in canary_grad_norms]
            
            # Count correct guesses
            correct = sum(1 for i in range(m) if guesses[i] == canary_in_mask[i])
            
            print(f"  Correct guesses: {correct}/{m} ({100*correct/m:.1f}%)")
            
            # Score = number of correct guesses (as per paper)
            if num_in > num_out:
                scores_in.append(correct)
                print(f"  ✓ Score (more IN): {correct}")
            else:
                scores_out.append(correct)
                print(f"  ✗ Score (more OUT): {correct}")
        
        del shadow
        torch.cuda.empty_cache()
    
    # Compute epsilon bound using Corollary 5.4
    scores_in = np.array(scores_in)
    scores_out = np.array(scores_out)
    
    print(f"\n{'='*60}")
    print("WHITE-BOX RESULTS")
    print(f"{'='*60}")
    print(f"Scores IN:  {scores_in}")
    print(f"Scores OUT: {scores_out}")
    print(f"Mean IN:  {np.mean(scores_in):.2f} ± {np.std(scores_in):.2f}")
    print(f"Mean OUT: {np.mean(scores_out):.2f} ± {np.std(scores_out):.2f}")
    
    if len(scores_in) == 0 or len(scores_out) == 0:
        print("⚠ Need both IN and OUT batches")
        return 0.0, 0.0, 0.0
    
    # Use the formula from Corollary 5.4
    r = m  # number of guesses
    v = np.mean(scores_in)  # average correct when canaries IN
    
    # For 95% confidence, find epsilon such that p-value = 0.05
    def p_value(eps_test):
        q = np.exp(eps_test) / (np.exp(eps_test) + 1)
        # Binomial probability
        from scipy.stats import binom
        p_val = 1 - binom.cdf(v - 1, r, q)
        # Add delta term (simplified from Theorem 5.2)
        p_val += 2 * m * delta / v if v > 0 else 2 * m * delta
        return min(p_val, 1.0)
    
    # Binary search for epsilon
    eps_min, eps_max = 0, 10
    for _ in range(30):
        eps_test = (eps_min + eps_max) / 2
        if p_value(eps_test) < 0.05:
            eps_min = eps_test
        else:
            eps_max = eps_test
    
    eps_lower_bound = eps_min
    
    print(f"\n📊 White‑box audit results (Corollary 5.4):")
    print(f"  Canaries: {m}, Guesses: {r}")
    print(f"  Avg correct (IN): {np.mean(scores_in):.1f}/{r} ({100*np.mean(scores_in)/r:.1f}%)")
    print(f"  Avg correct (OUT): {np.mean(scores_out):.1f}/{r} ({100*np.mean(scores_out)/r:.1f}%)")
    print(f"  ε ≥ {eps_lower_bound:.3f} (δ = {delta}, 95% confidence)")
    
    return eps_lower_bound, np.mean(scores_in)/r, np.mean(scores_out)/r


# ============================================================
# RUN STANDALONE AUDIT (No retraining)
# ============================================================

if __name__ == "__main__":
    print("Loading existing models (no retraining)...")
    
    # Load existing models
    baseline_mae, baseline_model = train_baseline(epochs=10, force_retrain=False)
    dp_mae, dp_eps, dp_model = train_dp(epochs=10, noise_multiplier=1.0, force_retrain=False)
    
    print(f"\nBaseline MAE: {baseline_mae:.3f} years")
    print(f"DP MAE: {dp_mae:.3f} years (claimed ε = {dp_eps:.2f})")
    
    # Run white-box audit only
    white_eps, white_acc_in, white_acc_out = white_box_audit_with_canaries(
        train_loader, 
        n_shadows=8,   # Number of shadow models
        epochs=3,      # Epochs per shadow model
        delta=1e-5
    )
    
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"Claimed ε: {dp_eps:.2f}")
    print(f"White-box ε ≥ {white_eps:.3f}")
    
    if white_eps > dp_eps:
        print("⚠ WHITE-BOX AUDIT SUGGESTS PRIVACY IS WORSE THAN CLAIMED!")
    elif white_eps > 0:
        print("✓ White-box audit consistent with claimed privacy")
    else:
        print("ℹ White-box audit cannot distinguish members (strong privacy)")

Loading existing models (no retraining)...

✓ Loading pre‑trained baseline model...
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Validation MAE: 5.102

✓ Loading pre‑trained DP model...
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Validation MAE: 13.281, ε = 0.37

Baseline MAE: 5.102 years
DP MAE: 13.281 years (claimed ε = 0.37)

WHITE‑BOX AUDIT (Steinke et al. 2023, Algorithm 3)
Gradient Space Attack with Dirac Canaries
Using 50 canaries
Will train 8 shadow models for 3 epochs each

[Shadow 1/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canaries IN: 25, OUT: 25
    Epoch 1/3: Avg grad norm = 0.000000
    Epoch 2/3: Avg grad norm = 0.000000
    Epoch 3/3: Avg grad norm = 0.000000
  Correct guesses: 25/50 (50.0%)
  ✗ Score (more OUT): 25

[Shadow 2/8]
✓ Transferred 60 pretrained weight layers
✓ Opacus validation passed
  Canaries IN: 26, OUT: 24
    Epoch 1/3: Avg grad norm = 0.000000
    Epoch 2/3: Avg gra